In [20]:
## Preprocessing
#### Needed packages
!pip install torch_geometric

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statistics
from google.colab import drive
import os
import torch
import torch.nn as nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv, Linear
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder


drive.mount('/content/drive')

## Loading data

# Listing contents of the mounted drive to help debug the path
print(os.listdir('/content/drive/'))

os.chdir ('/content/drive/MyDrive/MyProjects/GBDProject/Transformer(Graph)')

### Listing contents of the current directory to verify data.csv
print(os.listdir('.'))

df1 = pd.read_csv('GBD_DataAll.csv')
df2 = pd.read_csv('SDI_GBD.csv')
df1.head()
df2.head()

## Merging All the data
BigGBD=pd.merge(df1,df2,on=['location','year', 'sex'])
BigGBD

### Select8ng only variables of interest"
MeBigData=BigGBD[["location","sex","age_x","cause",	"risk_factor",	"year",
                  "deathratevalue","SDI_Quintile"]].copy()
MeBigData

MeBigData[["risk_factor"]].value_counts()

print(f"Loaded data shape: {MeBigData.shape}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Othercomputers', 'MyDrive', '.shortcut-targets-by-id', '.Trash-0', '.Encrypted']
['SDI_GBD.csv', 'GBD_DataAll.csv', 'GBDTransformers.ipynb']
Loaded data shape: (23630, 8)


In [25]:
# =========== HETEROGENEOUS GRAPH TRANSFORMER ============
#  (SDI ONLY, PER DISEASE)
# WITH SPATIAL SPILLOVER + TRAINING LOSS TRACKING
# ========================================================

import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv, Linear
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

# ====================== DATA ======================
countries = ["Burundi","Kenya","Rwanda","Tanzania","Uganda"]

df = BigGBD[BigGBD['location'].isin(countries)][
    ["location","age_x","cause","year","deathratevalue","SDI_Quintile"]
].dropna().copy()

le_country = LabelEncoder()
le_sdi     = LabelEncoder()

df['country_id'] = le_country.fit_transform(df['location'])
df['sdi_id']     = le_sdi.fit_transform(df['SDI_Quintile'])

def age_mid(x):
    if isinstance(x,str):
        if '+' in x:
            return float(x.replace('+',''))
        try:
            return np.mean([float(i) for i in x.split('-')])
        except:
            return 0.0
    return float(x)

df['age_mid'] = df['age_x'].apply(age_mid)

print("Countries:", sorted(le_country.classes_))
print("Total rows:", len(df))

# ====================== GRAPH ======================
def build_graph(df, spatial=False):
    data = HeteroData()

    data['country'].x = torch.eye(len(le_country.classes_))

    edge_index = torch.tensor([
        df['country_id'].values,
        df['country_id'].values
    ], dtype=torch.long)

    data['country','self','country'].edge_index = edge_index

    edge_attr = np.stack([
        df['sdi_id'].values.astype(np.float32),
        df['age_s'].values.astype(np.float32),
        df['year_s'].values.astype(np.float32)
    ], axis=1)

    data['country','self','country'].edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    data['country','self','country'].y = torch.tensor(df['y'].values, dtype=torch.float).unsqueeze(1)

    if spatial:
        neighbors = {
            "Uganda":["Kenya","Rwanda","Tanzania"],
            "Rwanda":["Uganda","Burundi","Tanzania"],
            "Kenya":["Uganda","Tanzania"],
            "Tanzania":["Uganda","Kenya","Rwanda","Burundi"],
            "Burundi":["Rwanda","Tanzania"]
        }

        edges=[]
        for s,ds in neighbors.items():
            for d in ds:
                edges.append([
                    le_country.transform([s])[0],
                    le_country.transform([d])[0]
                ])

        data['country','neighbor','country'].edge_index = torch.tensor(edges).t().contiguous()

    return data

# ====================== MODEL ======================
class SpilloverHGT(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.hidden = hidden
        self.proj = None
        self.convs = nn.ModuleList()
        self.mlp = None

    def init(self, data, device):
        if self.proj is not None:
            return

        self.proj = nn.ModuleDict({
            n: Linear(data[n].x.size(1), self.hidden).to(device)
            for n in data.node_types
        })

        metadata = data.metadata()

        for _ in range(2):
            self.convs.append(HGTConv(self.hidden, self.hidden, metadata, heads=4).to(device))

        edge_dim = data['country','self','country'].edge_attr.size(1)

        self.mlp = nn.Sequential(
            nn.Linear(self.hidden + edge_dim, self.hidden),
            nn.ReLU(),
            nn.Linear(self.hidden, 1)
        ).to(device)

    def forward(self, data):
        device = data['country','self','country'].y.device
        self.init(data, device)

        x = {k: self.proj[k](data[k].x) for k in self.proj}

        for conv in self.convs:
            x = conv(x, data.edge_index_dict)
            x = {k: F.relu(v) for k,v in x.items()}

        ei = data['country','self','country'].edge_index
        ea = data['country','self','country'].edge_attr
        src, _ = ei

        return self.mlp(torch.cat([x['country'][src], ea], dim=1))

# ====================== TRAIN ======================
def train_model(train_data, test_data, scaler, name="Model", epochs=200):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = SpilloverHGT().to(device)
    train_data = train_data.to(device)
    test_data  = test_data.to(device)

    _ = model(train_data)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loss_fn = nn.MSELoss()

    print(f"\n{name} Training...")

    final_loss = None

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        pred = model(train_data)
        loss = loss_fn(pred, train_data['country','self','country'].y)

        loss.backward()
        optimizer.step()

        final_loss = loss.item()

        if epoch % 50 == 0 or epoch == epochs-1:
            print(f"Epoch {epoch:3d} | Loss: {final_loss:.4f}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        pred_std = model(test_data).cpu().numpy().flatten()
        target_std = test_data['country','self','country'].y.cpu().numpy().flatten()

        pred_orig = scaler.inverse_transform(pred_std.reshape(-1, 1)).flatten()
        target_orig = scaler.inverse_transform(target_std.reshape(-1, 1)).flatten()

        mse = mean_squared_error(target_orig, pred_orig)
        r2 = r2_score(target_orig, pred_orig)

    return mse, r2, final_loss

# ====================== RUN ======================
diseases = sorted(df['cause'].unique())
results = []

for d in diseases:
    print("\n" + "="*70)
    print(f"Disease: {d}")
    print("="*70)

    df_d = df[df['cause'] == d].copy()

    train_df = df_d[df_d['year'] <= 2015].copy()
    test_df  = df_d[df_d['year'] > 2015].copy()

    target_scaler = StandardScaler()
    feat_scaler   = StandardScaler()

    train_df['y'] = target_scaler.fit_transform(train_df[['deathratevalue']])
    feat_scaler.fit(train_df[['age_mid','year']])

    train_df[['age_s','year_s']] = feat_scaler.transform(train_df[['age_mid','year']])
    test_df['y'] = target_scaler.transform(test_df[['deathratevalue']])
    test_df[['age_s','year_s']] = feat_scaler.transform(test_df[['age_mid','year']])

    train_base = build_graph(train_df, spatial=False)
    train_full = build_graph(train_df, spatial=True)
    test_base  = build_graph(test_df,  spatial=False)
    test_full  = build_graph(test_df,  spatial=True)

    mse_base, r2_base, loss_base = train_model(train_base, test_base, target_scaler, "Baseline")
    mse_full, r2_full, loss_full = train_model(train_full, test_full, target_scaler, "Spatial")

    improvement = (mse_base - mse_full) / mse_base * 100 if mse_base > 0 else 0

    # Interpretation
    if improvement > 8:
        interpretation = "Strong spillover"
        message = "→ Strong evidence of spatial spillover effects."
    elif improvement > 3:
        interpretation = "Moderate spillover"
        message = "→ Moderate evidence of spatial spillover."
    else:
        interpretation = "Weak / no spillover"
        message = "→ Weak or no clear spatial spillover detected."

    print(f"Improvement: {improvement:.2f}%")
    print(message)

    results.append({
        "Disease": d,
        "TrainLoss_Base": loss_base,
        "TrainLoss_Spatial": loss_full,
        "R2_Base": r2_base,
        "R2_Spatial": r2_full,
        "MSE_Base": mse_base,
         "MSE_Spatial":mse_full,
        "Improvement_%": improvement,
        "Spillover_Evidence": interpretation
    })

# ====================== FINAL ======================
results_df = pd.DataFrame(results)

print("\n" + "="*85)
print("FINAL RESULTS: SDI MODEL (WITH TRAINING LOSS)")
print(results_df)
print("="*85)



Countries: ['Burundi', 'Kenya', 'Rwanda', 'Tanzania', 'Uganda']
Total rows: 23630

Disease: HHD

Baseline Training...
Epoch   0 | Loss: 1.0235
Epoch  50 | Loss: 0.6916
Epoch 100 | Loss: 0.5791
Epoch 150 | Loss: 0.5242
Epoch 199 | Loss: 0.5063

Spatial Training...
Epoch   0 | Loss: 1.0280
Epoch  50 | Loss: 0.7308
Epoch 100 | Loss: 0.7081
Epoch 150 | Loss: 0.6678
Epoch 199 | Loss: 0.6250
Improvement: -26.08%
→ Weak or no clear spatial spillover detected.

Disease: IHD

Baseline Training...
Epoch   0 | Loss: 1.0136
Epoch  50 | Loss: 0.7936
Epoch 100 | Loss: 0.7479
Epoch 150 | Loss: 0.7208
Epoch 199 | Loss: 0.7138

Spatial Training...
Epoch   0 | Loss: 0.9972
Epoch  50 | Loss: 0.8000
Epoch 100 | Loss: 0.7543
Epoch 150 | Loss: 0.7192
Epoch 199 | Loss: 0.7133
Improvement: 0.37%
→ Weak or no clear spatial spillover detected.

Disease: diabetes

Baseline Training...
Epoch   0 | Loss: 1.0285
Epoch  50 | Loss: 0.6330
Epoch 100 | Loss: 0.5555
Epoch 150 | Loss: 0.5283
Epoch 199 | Loss: 0.5241

Spa

In [33]:
#### RISK ONLY, PER DISEASE ####
# FIXED RISK FACTOR: BMI, FPG, SBP
# ========================================================

import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv, Linear
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

# ====================== DATA ======================
countries = ["Burundi","Kenya","Rwanda","Tanzania","Uganda"]
risk_list = ["BMI", "FPG", "SBP"]  # FIXED ORDER

df = BigGBD[BigGBD['location'].isin(countries)][
    ["location","age_x","cause","year","deathratevalue","risk_factor"]
].dropna().copy()

# Keep only valid risk factors
df = df[df['risk_factor'].isin(risk_list)].copy()

# Encode country
le_country = LabelEncoder()
df['country_id'] = le_country.fit_transform(df['location'])

# ====================== MANUAL ONE-HOT (CLEAN) ======================
for r in risk_list:
    df[f"risk_{r}"] = (df['risk_factor'] == r).astype(float)

risk_cols = [f"risk_{r}" for r in risk_list]

# ====================== AGE ======================
def age_mid(x):
    if isinstance(x,str):
        if '+' in x:
            return float(x.replace('+',''))
        try:
            return np.mean([float(i) for i in x.split('-')])
        except:
            return 0.0
    return float(x)

df['age_mid'] = df['age_x'].apply(age_mid)

print("Countries:", sorted(le_country.classes_))
print("Risk factors used:", risk_list)
print("Total rows:", len(df))

# ====================== GRAPH ======================
def build_graph(df, spatial=False):
    data = HeteroData()

    data['country'].x = torch.eye(len(le_country.classes_))

    edge_index = torch.tensor([
        df['country_id'].values,
        df['country_id'].values
    ], dtype=torch.long)

    data['country','self','country'].edge_index = edge_index

    # EDGE FEATURES: risk (clean) + age + time
    edge_attr = np.column_stack([
        df[risk_cols].values.astype(np.float32),
        df['age_s'].values.astype(np.float32),
        df['year_s'].values.astype(np.float32)
    ])

    data['country','self','country'].edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    data['country','self','country'].y = torch.tensor(df['y'].values, dtype=torch.float).unsqueeze(1)

    if spatial:
        neighbors = {
            "Uganda":["Kenya","Rwanda","Tanzania"],
            "Rwanda":["Uganda","Burundi","Tanzania"],
            "Kenya":["Uganda","Tanzania"],
            "Tanzania":["Uganda","Kenya","Rwanda","Burundi"],
            "Burundi":["Rwanda","Tanzania"]
        }

        edges=[]
        for s,ds in neighbors.items():
            for d in ds:
                edges.append([
                    le_country.transform([s])[0],
                    le_country.transform([d])[0]
                ])

        data['country','neighbor','country'].edge_index = torch.tensor(edges).t().contiguous()

    return data

# ====================== MODEL ======================
class SpilloverHGT(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.hidden = hidden
        self.proj = None
        self.convs = nn.ModuleList()
        self.mlp = None

    def init(self, data, device):
        if self.proj is not None:
            return

        self.proj = nn.ModuleDict({
            n: Linear(data[n].x.size(1), self.hidden).to(device)
            for n in data.node_types
        })

        metadata = data.metadata()

        for _ in range(2):
            self.convs.append(HGTConv(self.hidden, self.hidden, metadata, heads=4).to(device))

        edge_dim = data['country','self','country'].edge_attr.size(1)

        self.mlp = nn.Sequential(
            nn.Linear(self.hidden + edge_dim, self.hidden),
            nn.ReLU(),
            nn.Linear(self.hidden, 1)
        ).to(device)

    def forward(self, data):
        device = data['country','self','country'].y.device
        self.init(data, device)

        x = {k: self.proj[k](data[k].x) for k in self.proj}

        for conv in self.convs:
            x = conv(x, data.edge_index_dict)
            x = {k: F.relu(v) for k,v in x.items()}

        ei = data['country','self','country'].edge_index
        ea = data['country','self','country'].edge_attr
        src, _ = ei

        return self.mlp(torch.cat([x['country'][src], ea], dim=1))

# ====================== TRAIN ======================
def train_model(train_data, test_data, scaler, name="Model", epochs=200):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = SpilloverHGT().to(device)
    train_data = train_data.to(device)
    test_data  = test_data.to(device)

    _ = model(train_data)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loss_fn = nn.MSELoss()

    print(f"\n{name} Training...")

    final_loss = None

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        pred = model(train_data)
        loss = loss_fn(pred, train_data['country','self','country'].y)

        loss.backward()
        optimizer.step()

        final_loss = loss.item()

        if epoch % 50 == 0 or epoch == epochs-1:
            print(f"Epoch {epoch:3d} | Loss: {final_loss:.4f}")

    model.eval()
    with torch.no_grad():
        pred_std = model(test_data).cpu().numpy().flatten()
        target_std = test_data['country','self','country'].y.cpu().numpy().flatten()

        pred_orig = scaler.inverse_transform(pred_std.reshape(-1, 1)).flatten()
        target_orig = scaler.inverse_transform(target_std.reshape(-1, 1)).flatten()

        mse = mean_squared_error(target_orig, pred_orig)
        r2 = r2_score(target_orig, pred_orig)

    return mse, r2, final_loss

# ====================== RUN ======================
diseases = sorted(df['cause'].unique())
results = []

for d in diseases:
    print("\n" + "="*70)
    print(f"Disease: {d}")
    print("="*70)

    df_d = df[df['cause'] == d].copy()

    train_df = df_d[df_d['year'] <= 2015].copy()
    test_df  = df_d[df_d['year'] > 2015].copy()

    target_scaler = StandardScaler()
    feat_scaler   = StandardScaler()

    train_df['y'] = target_scaler.fit_transform(train_df[['deathratevalue']])
    feat_scaler.fit(train_df[['age_mid','year']])

    train_df[['age_s','year_s']] = feat_scaler.transform(train_df[['age_mid','year']])
    test_df['y'] = target_scaler.transform(test_df[['deathratevalue']])
    test_df[['age_s','year_s']] = feat_scaler.transform(test_df[['age_mid','year']])

    train_base = build_graph(train_df, spatial=False)
    train_full = build_graph(train_df, spatial=True)
    test_base  = build_graph(test_df,  spatial=False)
    test_full  = build_graph(test_df,  spatial=True)

    mse_base, r2_base, loss_base = train_model(train_base, test_base, target_scaler, "Baseline")
    mse_full, r2_full, loss_full = train_model(train_full, test_full, target_scaler, "Spatial")

    improvement = (mse_base - mse_full) / mse_base * 100 if mse_base > 0 else 0

    if improvement > 8:
        interpretation = "Strong spillover"
    elif improvement > 3:
        interpretation = "Moderate spillover"
    else:
        interpretation = "Weak / no spillover"

    print(f"Improvement: {improvement:.2f}% → {interpretation}")

    results.append({
        "Disease": d,
        "TrainLoss_Base": loss_base,
        "TrainLoss_Spatial": loss_full,
        "R2_Base": r2_base,
        "R2_Spatial": r2_full,
        "MSE_Base": mse_base,
         "MSE_Spatial":mse_full,
        "Improvement_%": improvement,
        "Spillover_Evidence": interpretation
    })

# ====================== FINAL ======================
results_df = pd.DataFrame(results)

print("\n" + "="*85)
print("FINAL RESULTS: RISK-ONLY MODEL (BMI, FPG, SBP)")
print(results_df)
print("="*85)


Countries: ['Burundi', 'Kenya', 'Rwanda', 'Tanzania', 'Uganda']
Risk factors used: ['BMI', 'FPG', 'SBP']
Total rows: 23630

Disease: HHD

Baseline Training...
Epoch   0 | Loss: 0.9970
Epoch  50 | Loss: 0.5664
Epoch 100 | Loss: 0.3187
Epoch 150 | Loss: 0.1739
Epoch 199 | Loss: 0.1161

Spatial Training...
Epoch   0 | Loss: 1.0229
Epoch  50 | Loss: 0.5721
Epoch 100 | Loss: 0.4100
Epoch 150 | Loss: 0.2926
Epoch 199 | Loss: 0.1992
Improvement: -57.97% → Weak / no spillover

Disease: IHD

Baseline Training...
Epoch   0 | Loss: 1.0281
Epoch  50 | Loss: 0.5655
Epoch 100 | Loss: 0.3568
Epoch 150 | Loss: 0.1310
Epoch 199 | Loss: 0.0708

Spatial Training...
Epoch   0 | Loss: 1.0187
Epoch  50 | Loss: 0.5080
Epoch 100 | Loss: 0.1564
Epoch 150 | Loss: 0.0608
Epoch 199 | Loss: 0.0368
Improvement: 35.26% → Strong spillover

Disease: diabetes

Baseline Training...
Epoch   0 | Loss: 0.9639
Epoch  50 | Loss: 0.4631
Epoch 100 | Loss: 0.4248
Epoch 150 | Loss: 0.3235
Epoch 199 | Loss: 0.2186

Spatial Traini

In [ ]:
###################### FINAL CLEAN MODEL ######################
                  ###(Risk + SDI + Spatial) ###
# (country + disease + risk + SDI + age + time) → death rate

# ========================================================
# HETEROGENEOUS GRAPH TRANSFORMER (FULL MODEL)
# (country + SDI + risk + age + time) → death rate
# Spatial Spillover Test + Training Loss + Interpretation
# ========================================================

import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv, Linear
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

# ====================== DATA ======================
countries = ["Burundi","Kenya","Rwanda","Tanzania","Uganda"]
risk_list = ["BMI","FPG","SBP"]

df = BigGBD[BigGBD['location'].isin(countries)][
    ["location","age_x","cause","year","deathratevalue","SDI_Quintile","risk_factor"]
].dropna().copy()

# Keep valid risks
df = df[df['risk_factor'].isin(risk_list)].copy()

# Encode country + SDI
le_country = LabelEncoder()
le_sdi     = LabelEncoder()

df['country_id'] = le_country.fit_transform(df['location'])
df['sdi_id']     = le_sdi.fit_transform(df['SDI_Quintile'])

# Manual one-hot risk
for r in risk_list:
    df[f"risk_{r}"] = (df['risk_factor'] == r).astype(float)

risk_cols = [f"risk_{r}" for r in risk_list]

# Age midpoint
def age_mid(x):
    if isinstance(x,str):
        if '+' in x:
            return float(x.replace('+',''))
        try:
            return np.mean([float(i) for i in x.split('-')])
        except:
            return 0.0
    return float(x)

df['age_mid'] = df['age_x'].apply(age_mid)

print("Countries:", sorted(le_country.classes_))
print("Risk factors:", risk_list)
print("Total rows:", len(df))

# ====================== GRAPH ======================
def build_graph(df, spatial=False):
    data = HeteroData()

    data['country'].x = torch.eye(len(le_country.classes_))

    edge_index = torch.tensor([
        df['country_id'].values,
        df['country_id'].values
    ], dtype=torch.long)

    data['country','self','country'].edge_index = edge_index

    # EDGE FEATURES: SDI + risk + age + time
    edge_attr = np.column_stack([
        df['sdi_id'].values.astype(np.float32),
        df[risk_cols].values.astype(np.float32),
        df['age_s'].values.astype(np.float32),
        df['year_s'].values.astype(np.float32)
    ])

    data['country','self','country'].edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    data['country','self','country'].y = torch.tensor(df['y'].values, dtype=torch.float).unsqueeze(1)

    if spatial:
        neighbors = {
            "Uganda":["Kenya","Rwanda","Tanzania"],
            "Rwanda":["Uganda","Burundi","Tanzania"],
            "Kenya":["Uganda","Tanzania"],
            "Tanzania":["Uganda","Kenya","Rwanda","Burundi"],
            "Burundi":["Rwanda","Tanzania"]
        }

        edges=[]
        for s,ds in neighbors.items():
            for d in ds:
                edges.append([
                    le_country.transform([s])[0],
                    le_country.transform([d])[0]
                ])

        data['country','neighbor','country'].edge_index = torch.tensor(edges).t().contiguous()

    return data

# ====================== MODEL ======================
class SpilloverHGT(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.hidden = hidden
        self.proj = None
        self.convs = nn.ModuleList()
        self.mlp = None

    def init(self, data, device):
        if self.proj is not None:
            return

        self.proj = nn.ModuleDict({
            n: Linear(data[n].x.size(1), self.hidden).to(device)
            for n in data.node_types
        })

        metadata = data.metadata()

        for _ in range(2):
            self.convs.append(HGTConv(self.hidden, self.hidden, metadata, heads=4).to(device))

        edge_dim = data['country','self','country'].edge_attr.size(1)

        self.mlp = nn.Sequential(
            nn.Linear(self.hidden + edge_dim, self.hidden),
            nn.ReLU(),
            nn.Linear(self.hidden, 1)
        ).to(device)

    def forward(self, data):
        device = data['country','self','country'].y.device
        self.init(data, device)

        x = {k: self.proj[k](data[k].x) for k in self.proj}

        for conv in self.convs:
            x = conv(x, data.edge_index_dict)
            x = {k: F.relu(v) for k,v in x.items()}

        ei = data['country','self','country'].edge_index
        ea = data['country','self','country'].edge_attr
        src, _ = ei

        return self.mlp(torch.cat([x['country'][src], ea], dim=1))

# ====================== TRAIN ======================
def train_model(train_data, test_data, scaler, name="Model", epochs=200):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = SpilloverHGT().to(device)
    train_data = train_data.to(device)
    test_data  = test_data.to(device)

    _ = model(train_data)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loss_fn = nn.MSELoss()

    print(f"\n{name} Training...")

    final_loss = None

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        pred = model(train_data)
        loss = loss_fn(pred, train_data['country','self','country'].y)

        loss.backward()
        optimizer.step()

        final_loss = loss.item()

        if epoch % 50 == 0 or epoch == epochs-1:
            print(f"Epoch {epoch:3d} | Loss: {final_loss:.4f}")

    model.eval()
    with torch.no_grad():
        pred_std = model(test_data).cpu().numpy().flatten()
        target_std = test_data['country','self','country'].y.cpu().numpy().flatten()

        pred_orig = scaler.inverse_transform(pred_std.reshape(-1, 1)).flatten()
        target_orig = scaler.inverse_transform(target_std.reshape(-1, 1)).flatten()

        mse = mean_squared_error(target_orig, pred_orig)
        r2 = r2_score(target_orig, pred_orig)

    return mse, r2, final_loss

# ====================== RUN ======================
diseases = sorted(df['cause'].unique())
results = []

for d in diseases:
    print("\n" + "="*70)
    print(f"Disease: {d}")
    print("="*70)

    df_d = df[df['cause'] == d].copy()

    train_df = df_d[df_d['year'] <= 2015].copy()
    test_df  = df_d[df_d['year'] > 2015].copy()

    target_scaler = StandardScaler()
    feat_scaler   = StandardScaler()

    train_df['y'] = target_scaler.fit_transform(train_df[['deathratevalue']])
    feat_scaler.fit(train_df[['age_mid','year']])

    train_df[['age_s','year_s']] = feat_scaler.transform(train_df[['age_mid','year']])
    test_df['y'] = target_scaler.transform(test_df[['deathratevalue']])
    test_df[['age_s','year_s']] = feat_scaler.transform(test_df[['age_mid','year']])

    train_base = build_graph(train_df, spatial=False)
    train_full = build_graph(train_df, spatial=True)
    test_base  = build_graph(test_df,  spatial=False)
    test_full  = build_graph(test_df,  spatial=True)

    mse_base, r2_base, loss_base = train_model(train_base, test_base, target_scaler, "Baseline")
    mse_full, r2_full, loss_full = train_model(train_full, test_full, target_scaler, "Spatial")

    improvement = (mse_base - mse_full) / mse_base * 100 if mse_base > 0 else 0

    if improvement > 8:
        interpretation = "Strong spillover"
    elif improvement > 3:
        interpretation = "Moderate spillover"
    else:
        interpretation = "Weak / no spillover"

    print(f"Improvement: {improvement:.2f}% → {interpretation}")

    results.append({
        "Disease": d,
        "TrainLoss_Base": loss_base,
        "TrainLoss_Spatial": loss_full,
        "R2_Base": r2_base,
        "R2_Spatial": r2_full,
        "MSE_Base": mse_base,
         "MSE_Spatial":mse_full,
        "Improvement_%": improvement,
        "Spillover_Evidence": interpretation
    })

# ====================== FINAL ======================
results_df = pd.DataFrame(results)

print("\n" + "="*85)
print("FINAL RESULTS: FULL MODEL (SDI + RISK)")
print(results_df)
print("="*85)


